# Cross-Dataset Probe Transferability

Train a probe on one dataset's activations and evaluate on another's.
Produces an N×N accuracy matrix (rows = train, columns = test).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
JUDGE_MODEL      = 'llama-3.1-8b'
EXTRACTION_MODEL = 'gemma-3-27b'

# One extraction date per dataset, positionally aligned
DATASETS          = ['pond', 'nfix']
EXTRACTION_DATES  = ['2026_04_01', '2026_04_01']

In [ ]:
import pandas as pd
from analysis.cross_dataset import cross_dataset_probe_matrix
from analysis.plots import cross_dataset_matrix as plot_cross_dataset_matrix
import paths

## Build accuracy matrix

In [ ]:
df = cross_dataset_probe_matrix(
    judge_model=JUDGE_MODEL,
    datasets=DATASETS,
    extraction_model=EXTRACTION_MODEL,
    extraction_dates=EXTRACTION_DATES,
)
df.round(3)

## Heatmap

In [ ]:
fig = plot_cross_dataset_matrix(df)
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/cross_dataset_matrix.pdf', bbox_inches='tight')
fig.show()

## In-domain vs. cross-domain

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vals = df.values.astype(float)
in_domain    = np.nanmean(np.diag(vals))
cross_domain = np.nanmean(vals[~np.eye(len(vals), dtype=bool)])

fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(['In-domain', 'Cross-domain'], [in_domain, cross_domain], color=['steelblue', 'tomato'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Probe accuracy')
ax.set_title(f'{JUDGE_MODEL} / {EXTRACTION_MODEL}')
fig.tight_layout()
fig.savefig('figures/in_vs_cross_domain.pdf', bbox_inches='tight')
fig.show()

print(f'In-domain:    {in_domain:.4f}')
print(f'Cross-domain: {cross_domain:.4f}')

## Save results

In [ ]:
out_dir = paths.cross_dataset_dir()
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'probe_matrix_{JUDGE_MODEL}_{EXTRACTION_MODEL}.csv'
df.to_csv(out_path)
print(f'Saved to {out_path}')